In [1]:
pip install git+https://github.com/fluxnet/shuttle.git

  Cloning https://github.com/fluxnet/shuttle.git to /tmp/pip-req-build-x3m04s7e
  Running command git clone --filter=blob:none --quiet https://github.com/fluxnet/shuttle.git /tmp/pip-req-build-x3m04s7e
  Resolved https://github.com/fluxnet/shuttle.git to commit 028818791fef116db03e97f3f6b0aebdf1a89e90
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for fluxnet-shuttle: filename=fluxnet_shuttle-0.3.8-py3-none-any.whl size=55276 sha256=9bd8fcd9dce0a894b7eb85c2754c611bcd3ec10ba4c3d346c972fbd801b3e1be
  Stored in directory: /tmp/pip-ephem-wheel-cache-025drqk5/wheels/36/f3/39/4dc131af04263cfa244ad29098bef1077e7ea18716c06ead93
Successfully built fluxnet-shuttle
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [fluxnet-shuttle]
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
import pandas as pd
import subprocess
import glob
import os

# -----------------------------
# Settings
# -----------------------------
#out_dir = Path("/mnt/gsdata/projects/panops/panops-data-registry/data/flux/fluxnet_2017_2025_V02")
# for Daniel
out_dir = Path("/mnt/gsdata/projects/panops/panops-data-registry/data/flux/Flux4Daniel")
out_dir.mkdir(parents=True, exist_ok=True)

target_igbp = ["CSH", "OSH", "SAV", "WSA", "WET", "MF", "ENF", "EBF", "DNF", "DBF", "CRO", "CVM", "GRA", "BSV" ]

#start_year = 2017
#end_year = 2025

# Change this to 1, 3, 4, etc.
# Use 1 if you want any data between 2017 and 2025.
#min_continuous_years = 1

os.chdir(out_dir)
print("Working in:", out_dir)

Working in: /mnt/gsdata/projects/panops/panops-data-registry/data/flux/Flux4Daniel


In [ ]:
import glob
import os

# -----------------------------
# Settings
# -----------------------------
out_dir = Path("/mnt/gsdata/projects/panops/panops-data-registry/data/flux/fluxnet_2017_2025_V02")
out_dir.mkdir(parents=True, exist_ok=True)

target_igbp = ["CSH", "OSH", "SAV", "WSA", "WET", "MF", "ENF", "EBF", "DNF", "DBF"]

start_year = 2017
end_year = 2025

# Change this to 1, 3, 4, etc.
# Use 1 if you want any data between 2017 and 2025.
min_continuous_years = 1

os.chdir(out_dir)
print("Working in:", out_dir)

In [2]:
# -----------------------------
# Step 1: create latest Shuttle snapshot
# -----------------------------
subprocess.run(
    ["fluxnet-shuttle", "--verbose", "listall"],
    check=True
)

snapshots = sorted(out_dir.glob("fluxnet_shuttle_snapshot*.csv"))
snapshot_file = snapshots[-1]

snapshot = pd.read_csv(snapshot_file)

print("Using snapshot:", snapshot_file)
print("Snapshot rows:", len(snapshot))
print(snapshot.columns.tolist())

2026-05-22 09:21:02,966 - fluxnet_shuttle.main - DEBUG - FLUXNET SHUTTLE RUN started at 2026-05-22 09:21:02.951725
2026-05-22 09:21:02,966 - fluxnet_shuttle.main - DEBUG - Running listall command
2026-05-22 09:21:02,966 - asyncio - DEBUG - Using selector: EpollSelector
2026-05-22 09:21:02,967 - fluxnet_shuttle.shuttle - DEBUG - Data hubs to include: all available
2026-05-22 09:21:02,971 - fluxnet_shuttle.core.config - INFO - Loaded default configuration from package
2026-05-22 09:21:02,971 - fluxnet_shuttle.core.shuttle - INFO - Initialized FluxnetShuttle with data hubs: ['ameriflux', 'icos', 'tern']
2026-05-22 09:21:03,008 - fluxnet_shuttle.plugins.ameriflux - INFO - Fetching AmeriFlux sites...
2026-05-22 09:21:07,581 - fluxnet_shuttle.plugins.ameriflux - INFO - Retrieved metadata for 383 AmeriFlux sites
2026-05-22 09:21:08,594 - fluxnet_shuttle.plugins.ameriflux - INFO - Retrieved download links for 383 sites
2026-05-22 09:21:08,858 - fluxnet_shuttle.plugins.ameriflux - INFO - Retrie

In [3]:
# -----------------------------
# Step 2: filter by IGBP and years
# -----------------------------
metadata = snapshot.copy()

metadata["igbp"] = metadata["igbp"].astype(str).str.strip()
metadata["site_id"] = metadata["site_id"].astype(str).str.strip()

metadata = metadata[
    metadata["igbp"].isin(target_igbp)
].copy()

### for Daniel comment this part out
# Keep sites that overlap with 2017–2025
#metadata = metadata[
#    (metadata["first_year"] <= end_year) &
#    (metadata["last_year"] >= start_year)
#].copy()

metadata = metadata.drop_duplicates(subset="site_id")

site_list = metadata["site_id"].tolist()

print("Sites after IGBP filter:", len(metadata))

print("Number of sites to download:", len(site_list))

metadata[["site_id", "site_name", "igbp", "first_year", "last_year"]].head()

Sites after IGBP filter: 717
Number of sites to download: 717


,site_id,site_name,igbp,first_year,last_year
0,AR-Bal,Balcarce BA,CRO,2012,2013
1,AR-CCa,Carlos Casares agriculture,CRO,2012,2020
2,AR-CCg,Carlos Casares grassland,GRA,2018,2024
3,AR-TF1,Rio Moat bog,WET,2016,2018
4,AR-TF2,Rio Pipo bog,WET,2016,2018


In [4]:
metadata_file = out_dir / "selected_site_metadata.csv"
metadata.to_csv(metadata_file, index=False)

print("Saved metadata:", metadata_file)

Saved metadata: /mnt/gsdata/projects/panops/panops-data-registry/data/flux/Flux4Daniel/selected_site_metadata.csv


In [4]:
# -----------------------------
# Step 3: build site-year availability table
# -----------------------------
availability = []

for _, row in metadata.iterrows():
    available_start = max(int(row["first_year"]), start_year)
    available_end = min(int(row["last_year"]), end_year)
    
    for year in range(available_start, available_end + 1):
        availability.append({
            "site_id": row["site_id"],
            "igbp": row["igbp"],
            "year": year
        })

availability_df = pd.DataFrame(availability)

print("Availability rows:", len(availability_df))
availability_df.head()

Availability rows: 1914


,site_id,igbp,year
0,AR-TF1,WET,2017
1,AR-TF1,WET,2018
2,AR-TF2,WET,2017
3,AR-TF2,WET,2018
4,BR-ITA,EBF,2020


In [5]:
# -----------------------------
# Step 4: filter by continuous years
# -----------------------------
def max_consecutive_years(years):
    years = sorted(set(years))
    if not years:
        return 0
    
    max_run = 1
    current_run = 1
    
    for i in range(1, len(years)):
        if years[i] == years[i - 1] + 1:
            current_run += 1
            max_run = max(max_run, current_run)
        else:
            current_run = 1
            
    return max_run

site_continuity = (
    availability_df
    .groupby("site_id")["year"]
    .apply(max_consecutive_years)
    .reset_index(name="max_consecutive_years_2017_2025")
)

selected_sites_df = site_continuity[
    site_continuity["max_consecutive_years_2017_2025"] >= min_continuous_years
].copy()

selected_metadata = metadata.merge(
    selected_sites_df,
    on="site_id",
    how="inner"
)

site_list = selected_metadata["site_id"].tolist()

print(f"Selected sites with at least {min_continuous_years} continuous years:", len(site_list))

selected_metadata[
    ["site_id", "site_name", "igbp", "first_year", "last_year", "max_consecutive_years_2017_2025"]
].head()

Selected sites with at least 1 continuous years: 357


,site_id,site_name,igbp,first_year,last_year,max_consecutive_years_2017_2025
0,AR-TF1,Rio Moat bog,WET,2016,2018,2
1,AR-TF2,Rio Pipo bog,WET,2016,2018,2
2,BR-ITA,MataFLUX: Multi-species tree restoration planting,EBF,2020,2024,5
3,BR-Npw,Northern Pantanal Wetland,WSA,2013,2017,1
4,CA-ARB,Attawapiskat River Bog,WET,2011,2024,8


In [6]:
# -----------------------------
# Step 5: save selected metadata before downloading
# -----------------------------
selected_metadata_file = out_dir / f"selected_sites_igbp_{start_year}_{end_year}_min{min_continuous_years}yr.csv"
availability_file = out_dir / f"selected_sites_availability_{start_year}_{end_year}_min{min_continuous_years}yr.csv"

selected_metadata.to_csv(selected_metadata_file, index=False)
availability_df.to_csv(availability_file, index=False)

print("Saved:")
print(selected_metadata_file)
print(availability_file)

Saved:
/mnt/gsdata/projects/panops/panops-data-registry/data/flux/fluxnet_2017_2025_V02/selected_sites_igbp_2017_2025_min1yr.csv
/mnt/gsdata/projects/panops/panops-data-registry/data/flux/fluxnet_2017_2025_V02/selected_sites_availability_2017_2025_min1yr.csv


In [5]:
# -----------------------------
# Step 6: download only selected sites
# -----------------------------
cmd = [
    "fluxnet-shuttle",
    "--verbose",
    "download",
    "-f", str(snapshot_file),
    "-o", str(out_dir),
    "-s", *site_list,
]

result = subprocess.run(
    cmd,
    input="\n" * 20,
    text=True,
    check=True
)

print("Download finished.")

2026-05-22 09:36:18,352 - fluxnet_shuttle.main - DEBUG - FLUXNET SHUTTLE RUN started at 2026-05-22 09:36:18.347702
2026-05-22 09:36:18,354 - fluxnet_shuttle.main - DEBUG - Running download command with 717 site IDs: ['AR-Bal', 'AR-CCa', 'AR-CCg', 'AR-TF1', 'AR-TF2', 'BR-CST', 'BR-ITA', 'BR-Ji3', 'BR-Ma2', 'BR-Ma3', 'BR-Npw', 'BR-Sa1', 'BR-Sa3', 'BR-SM1', 'CA-ARB', 'CA-ARF', 'CA-BOU', 'CA-Ca1', 'CA-Ca2', 'CA-Ca3', 'CA-CF1', 'CA-CF2', 'CA-CF3', 'CA-DB2', 'CA-DBB', 'CA-DSM', 'CA-EM1', 'CA-EM2', 'CA-Gro', 'CA-HPC', 'CA-KLP', 'CA-LP1', 'CA-LU1', 'CA-LU2', 'CA-MA1', 'CA-MA2', 'CA-MA3', 'CA-Man', 'CA-Mer', 'CA-Na1', 'CA-PB1', 'CA-PB2', 'CA-Qc2', 'CA-Qfo', 'CA-RBM', 'CA-SCB', 'CA-SCC', 'CA-SF1', 'CA-SF3', 'CA-SMC', 'CA-TP1', 'CA-TP3', 'CA-TPA', 'CA-TPD', 'CL-ACF', 'CL-SDF', 'CL-SDP', 'CR-Fsc', 'MX-Aog', 'MX-PMm', 'MX-Tes', 'PE-QFR', 'PR-xLA', 'US-A10', 'US-A32', 'US-A37', 'US-A39', 'US-A74', 'US-ADR', 'US-Akn', 'US-ALQ', 'US-AR1', 'US-AR2', 'US-ARb', 'US-ARc', 'US-ASH', 'US-ASM', 'US-Bar', 'US